In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings("ignore")

# 1. 加载数据（只需修改这里↓↓↓）
data_path = r"D:\萝卜投稿专用\Original Research - Spinal Infection Following Internal Fixation Surgery\图表及其来源\KNN与MICE的选择，归一化\插值选择\训练集_原始数据.xlsx"
data = pd.read_excel(data_path)

# 确保数据中包含 Infection 列
if "Infection" not in data.columns:
    raise ValueError("数据中必须包含 'Infection' 列作为结局标签")

# Infection 是结局标签：不进入插补模型，只用于最后拼回/分组检查
infection_label = data["Infection"].copy()
feature_cols = [col for col in data.columns if col != "Infection"]
feature_data = data[feature_cols].copy()

# 2. 找出所有完整列：只在特征列中找，排除 Infection
complete_cols = [col for col in feature_cols if feature_data[col].isnull().sum() == 0]
print("完整特征列有:", complete_cols)

if not complete_cols:
    raise ValueError("没有找到完整特征列！")

# 3. 对每个完整特征列随机删除 10% 数据（制造缺失）
np.random.seed(42)  # 固定随机种子
data_with_missing = feature_data.copy()
missing_records = pd.DataFrame()  # 记录被删除的真实值

for col in complete_cols:
    missing_idx = np.random.choice(
        feature_data.index,
        size=int(len(feature_data) * 0.1),
        replace=False
    )
    # 记录被删除的值
    missing_records = pd.concat([
        missing_records,
        pd.DataFrame({
            "列名": col,
            "行索引": missing_idx,
            "真实值": feature_data.loc[missing_idx, col]
        })
    ])
    # 设置缺失值
    data_with_missing.loc[missing_idx, col] = np.nan


In [ ]:
# 4. XGBoost 插补参数筛选：Infection 不作为预测因子
# Phase 1

n_estimators_list = [
    50,
    100,
    200
]

learning_rates = [
    0.01,
    0.04,
    0.07,
    0.10,
    0.13,
    0.16,
    0.19,
    0.22,
    0.25,
    0.28
]

phase1_results = []

for n_est in n_estimators_list:

    for lr in learning_rates:

        print(
            f"\nTesting "
            f"n_estimators={n_est}, "
            f"learning_rate={lr}"
        )

        imputer = IterativeImputer(

            estimator=XGBRegressor(
                n_estimators=n_est,
                learning_rate=lr,
                random_state=42,
                n_jobs=-1
            ),

            max_iter=12,
            random_state=42
        )

        # 关键：这里只传入 data_with_missing；它已经不含 Infection
        imputed_data = imputer.fit_transform(
            data_with_missing
        )

        imputed_df = pd.DataFrame(
            imputed_data,
            columns=feature_cols,
            index=data_with_missing.index
        )

        nrmse_list = []

        for col in complete_cols:

            col_records = missing_records[
                missing_records["列名"] == col
            ]

            y_true = col_records["真实值"]

            y_pred = imputed_df.loc[
                col_records["行索引"],
                col
            ]

            rmse = np.sqrt(
                np.mean(
                    (y_true - y_pred) ** 2
                )
            )

            nrmse = rmse / (
                y_true.max() - y_true.min()
            )

            nrmse_list.append(nrmse)

        mean_nrmse = np.mean(
            nrmse_list
        )
        sd_nrmse = np.std(
            nrmse_list,
            ddof=1
        )

        phase1_results.append({
            "n_estimators": n_est,
            "learning_rate": lr,
            "Mean_NRMSE": mean_nrmse,
            "SD_NRMSE": sd_nrmse
        })

        print(
            f"Mean NRMSE={mean_nrmse:.5f}, "
            f"SD NRMSE={sd_nrmse:.5f}"
        )

phase1_df = pd.DataFrame(
    phase1_results
)

phase1_df = phase1_df.sort_values(
    "Mean_NRMSE"
)

phase1_df.to_excel(
    "Phase1_All_Results_no_Infection_predictor.xlsx",
    index=False
)

print("\nTOP 3")
print(
    phase1_df.head(3)
)


In [ ]:
# Phase 2：对 TOP 参数进行多随机种子验证，Infection 不作为预测因子
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings("ignore")

# 1. 加载数据（只需修改这里↓↓↓）
data_path = r"D:\萝卜投稿专用\Original Research - Spinal Infection Following Internal Fixation Surgery\图表及其来源\KNN与MICE的选择，归一化\插值选择\训练集_原始数据.xlsx"
data = pd.read_excel(data_path)

# 确保数据中包含 Infection 列
if "Infection" not in data.columns:
    raise ValueError("数据中必须包含 'Infection' 列作为结局标签")

# Infection 是结局标签：不进入插补模型，只用于最后拼回/分组检查
infection_label = data["Infection"].copy()
feature_cols = [col for col in data.columns if col != "Infection"]
feature_data = data[feature_cols].copy()

# 2. 找出所有完整列：只在特征列中找，排除 Infection
complete_cols = [col for col in feature_cols if feature_data[col].isnull().sum() == 0]
print("完整特征列有:", complete_cols)

if not complete_cols:
    raise ValueError("没有找到完整特征列！")

# 3. 对每个完整特征列随机删除 10% 数据（制造缺失）
np.random.seed(42)  # 固定随机种子
data_with_missing = feature_data.copy()
missing_records = pd.DataFrame()  # 记录被删除的真实值

for col in complete_cols:
    missing_idx = np.random.choice(
        feature_data.index,
        size=int(len(feature_data) * 0.1),
        replace=False
    )
    # 记录被删除的值
    missing_records = pd.concat([
        missing_records,
        pd.DataFrame({
            "列名": col,
            "行索引": missing_idx,
            "真实值": feature_data.loc[missing_idx, col]
        })
    ])
    # 设置缺失值
    data_with_missing.loc[missing_idx, col] = np.nan

# 根据 Phase 1 的结果手动填写 TOP 参数
top3_params = [
    (200, 0.07),
    (100, 0.16),
    (100, 0.22)
]

phase2_results = []

for n_est, lr in top3_params:

    print("\n===================")
    print(
        f"Testing "
        f"n_estimators={n_est}, "
        f"lr={lr}"
    )
    print("===================")

    run_nrmse = []

    for seed in range(100, 1100, 100):

        print(f"Seed={seed}")

        imputer = IterativeImputer(

            estimator=XGBRegressor(
                n_estimators=n_est,
                learning_rate=lr,
                random_state=seed,
                n_jobs=-1
            ),

            max_iter=12,
            random_state=seed
        )

        # 关键：这里只传入 data_with_missing；它已经不含 Infection
        imputed_data = imputer.fit_transform(
            data_with_missing
        )

        imputed_df = pd.DataFrame(
            imputed_data,
            columns=feature_cols,
            index=data_with_missing.index
        )

        nrmse_list = []

        for col in complete_cols:

            col_records = missing_records[
                missing_records["列名"] == col
            ]

            y_true = col_records["真实值"]

            y_pred = imputed_df.loc[
                col_records["行索引"],
                col
            ]

            rmse = np.sqrt(
                np.mean(
                    (y_true - y_pred) ** 2
                )
            )

            nrmse = rmse / (
                y_true.max() - y_true.min()
            )

            nrmse_list.append(nrmse)

        mean_nrmse = np.mean(
            nrmse_list
        )

        run_nrmse.append(
            mean_nrmse
        )

    overall_mean = np.mean(
        run_nrmse
    )

    overall_sd = np.std(
        run_nrmse,
        ddof=1
    )

    ci_low = np.percentile(
        run_nrmse,
        2.5
    )

    ci_high = np.percentile(
        run_nrmse,
        97.5
    )

    phase2_results.append({
        "n_estimators": n_est,
        "learning_rate": lr,
        "Mean_NRMSE": overall_mean,
        "SD_NRMSE": overall_sd,
        "CI_Low": ci_low,
        "CI_High": ci_high
    })

phase2_df = pd.DataFrame(
    phase2_results
)

phase2_df = phase2_df.sort_values(
    "Mean_NRMSE"
)

phase2_df.to_excel(
    "Phase2_Final_Ranking_no_Infection_predictor.xlsx",
    index=False
)

print("\n===================")
print("FINAL RESULT")
print("===================")
print(
    phase2_df
)
